## Library Imports
## Importing necessary libraries: Dash for web interface, NumPy for numerical computations, Plotly for visualizations, Pyngrok for secure URLs.


In [1]:
!pip install dash plotly tensorflow numpy pandas scipy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 44.2 MB/s eta 0:00:00


In [2]:
!pip install dash
!pip install pyngrok

Installing Dash Bootstrap Components

In [3]:
!pip install dash-bootstrap-components

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.5/222.5 kB 4.0 MB/s eta 0:00:00


# HyVIS: TSP Visualization Tool Implementation


In [4]:
import dash
from dash import dcc
from dash import html
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output, State
import numpy as np
import plotly.graph_objects as go
from pyngrok import ngrok

# Placeholder implementation for the TSPProblemDomain class
class TSPProblemDomain:
    def __init__(self, cities=None, num_cities=10):
        if cities is not None:
            self.cities = cities
        else:
            self.cities = np.random.rand(num_cities, 2)
        self.num_cities = len(self.cities)
        self.solutions = [None] * 2

    def initialiseSolution(self, index):
        self.solutions[index] = np.random.permutation(self.num_cities)

    def applyHeuristic(self, heuristic_index, solution_index, result_index):
        new_solution = self.solutions[solution_index].copy()
        if heuristic_index == 0:
            i, j = np.random.choice(len(new_solution), 2, replace=False)
            new_solution[i], new_solution[j] = new_solution[j], new_solution[i]
        elif heuristic_index == 1:
            i, j = sorted(np.random.choice(len(new_solution), 2, replace=False))
            new_solution[i:j] = new_solution[i:j][::-1]
        self.solutions[result_index] = new_solution
        return new_solution

    def getFunctionValue(self, index):
        solution = self.solutions[index]
        return np.sum(np.linalg.norm(self.cities[solution] - np.roll(self.cities[solution], shift=-1, axis=0), axis=1))

    def getState(self):
        return [self.getFunctionValue(0)]

    def hasTimeExpired(self):
        return False

# RL agent with utility-based heuristic selection
class SimpleUtilityBasedAgent:
    def __init__(self, action_size, problem_domain, initial_utility=1.0, adaptation_strategy='additive'):
        self.action_size = action_size
        self.problem_domain = problem_domain
        self.utilities = {i: initial_utility for i in range(action_size)}
        self.utilization_counts = {i: 0 for i in range(action_size)}
        self.effective_utilization_counts = {i: 0 for i in range(action_size)}
        self.adaptation_strategy = adaptation_strategy
        self.reward_progress = []

    def select_action(self):
        action = max(self.utilities, key=self.utilities.get)
        self.utilization_counts[action] += 1
        return action

    def update_utility(self, action, improvement):
        if improvement:
            self.effective_utilization_counts[action] += 1
            if self.adaptation_strategy == 'additive':
                self.utilities[action] += 1
            elif self.adaptation_strategy == 'divisional':
                self.utilities[action] *= 1.1
            elif self.adaptation_strategy == 'root':
                self.utilities[action] = np.sqrt(self.utilities[action])
        else:
            if self.adaptation_strategy == 'additive':
                self.utilities[action] -= 1
            elif self.adaptation_strategy == 'divisional':
                self.utilities[action] *= 0.9
            elif self.adaptation_strategy == 'root':
                self.utilities[action] = np.sqrt(self.utilities[action])
        reward = self.utilities[action]
        self.reward_progress.append(reward)

# Training function
def train_utility_based_agent(agent, episodes, max_steps):
    performance_metrics = []
    reward_progress = []
    total_distances = []

    for episode in range(episodes):
        agent.problem_domain.initialiseSolution(0)
        total_distance = agent.problem_domain.getFunctionValue(0)

        for step in range(max_steps):
            action = agent.select_action()
            previous_cost = agent.problem_domain.getFunctionValue(0)
            agent.problem_domain.applyHeuristic(action, 0, 1)
            new_cost = agent.problem_domain.getFunctionValue(1)


            reward = previous_cost - new_cost

            improvement = new_cost < previous_cost
            agent.update_utility(action, improvement)

            if improvement:
                agent.problem_domain.solutions[0] = agent.problem_domain.solutions[1]

            # Record the reward for the reward progress plot
            reward_progress.append(reward)

            # Record the total distance after each step for the convergence plot
            performance_metrics.append(agent.problem_domain.getFunctionValue(0))

        final_distance = agent.problem_domain.getFunctionValue(0)
        total_distances.append(final_distance)  # Appending the total distance for each episode

    # Calculate the average distance across all episodes
    average_distance = np.mean(total_distances)

    return performance_metrics, reward_progress, average_distance


pastel_theme = {
    'paper_bgcolor': '#fafafa',
    'plot_bgcolor': '#fafafa',
    'font': {'color': '#555555', 'family': 'Arial'},
    'xaxis': {'gridcolor': '#e5e5e5', 'zerolinecolor': '#e5e5e5'},
    'yaxis': {'gridcolor': '#e5e5e5', 'zerolinecolor': '#e5e5e5'},
}


pastel_colors = {
    'bar': '#84a9ac',
    'bar_secondary': '#f6d7a7',
    'pie': ['#ffd3b6', '#a8e6cf']
}

# Visualization functions with pastel theme
def create_effective_utilization_bar_chart(effective_utilization, cooling_schedule_name):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=list(effective_utilization.keys()),
        y=list(effective_utilization.values()),
        marker=dict(color=[pastel_colors['bar'], pastel_colors['bar_secondary']]),
        name='Effective Utilisation'
    ))
    fig.update_layout(
        title=f'Effective Utilisation Rate of Heuristics<br> ({cooling_schedule_name} Cooling Schedule)',
        xaxis_title='Heuristic',
        yaxis_title='Effective Utilization Rate',
        **pastel_theme
    )
    return fig

def create_reward_progress_plot(reward_progress, cooling_schedule_name):
    fig = go.Figure(data=go.Scatter(x=list(range(len(reward_progress))),
                                    y=reward_progress,
                                    mode='lines'))
    fig.update_layout(
        title=f'Reward Progress ({cooling_schedule_name} Cooling Schedule)',
        xaxis_title='Iteration',
        yaxis_title='Reward Value'
    )
    return fig

def create_convergence_plot(convergence_data, cooling_schedule_name):
    fig = go.Figure(data=go.Scatter(x=list(range(len(convergence_data))),
                                    y=convergence_data,
                                    mode='lines'))
    fig.update_layout(
        title=f'Convergence Plot ({cooling_schedule_name} Cooling Schedule)',
        xaxis_title='Iteration',
        yaxis_title='Total Distance'
    )
    return fig

def create_utilization_pie_chart(utilization_rates, cooling_schedule_name):
    labels = [f"Heuristic {key}" for key in utilization_rates.keys()]
    values = list(utilization_rates.values())

    fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3, marker=dict(colors=pastel_colors['pie']))])
    fig.update_layout(
        title_text=f'Utilisation Rate of Heuristics ({cooling_schedule_name} Cooling Schedule)',
        **pastel_theme
    )
    return fig

def create_tsp_figure(route, cities):
    fig = go.Figure()

    x = np.append(cities[route, 0], cities[route[0], 0])
    y = np.append(cities[route, 1], cities[route[0], 1])

    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        name='Route',
        line=dict(color='red'),
        marker=dict(size=8, color='red')
    ))

    city_numbers = [str(i + 1) for i in range(len(route))]
    city_numbers.append("1")

    fig.add_trace(go.Scatter(
        x=cities[route, 0],
        y=cities[route, 1],
        mode='markers+text',
        name='Cities',
        marker=dict(color='blue'),
        text=city_numbers,
        textposition='top center'
    ))

    fig.update_layout(
        title='Optimized TSP Route',
        xaxis_title='X',
        yaxis_title='Y'
    )

    return fig

def calculate_total_distance(route, cities):
    distance = np.sum(np.linalg.norm(cities[route] - np.roll(cities[route], shift=-1, axis=0), axis=1))
    return distance

def get_utilization_rates(agent):
    total = sum(agent.utilization_counts.values())
    return {k: v / total for k, v in agent.utilization_counts.items()}

def get_effective_utilization_rates(agent):
    total_improvements = sum(agent.effective_utilization_counts.values())
    return {k: v / total_improvements for k, v in agent.effective_utilization_counts.items()}

# Defining the Bootstrap Icons CDN
bootstrap_icons = "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.10.5/font/bootstrap-icons.css"

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP, bootstrap_icons])

app.layout = html.Div([

    html.H1("HyVIS: TSP Visualisation Tool", style={'text-align': 'center', 'font-size': '32px', 'margin-bottom': '20px'}),

    # Parameter section
    html.Div([
        # TSP Parameters Section
        html.Div([
            html.H4('TSP Parameters', style={'text-align': 'center', 'font-size': '20px', 'margin-bottom': '15px'}),
            html.Label('Number of Cities', style={'font-weight': 'bold', 'font-size': '16px'}),
            dcc.Input(id='num-cities', type='number', value=10, style={'margin-bottom': '15px', 'width': '60px'}),
            html.Br(),
            html.Label('Episodes', style={'font-weight': 'bold', 'font-size': '16px'}),
            dcc.Input(id='episodes', type='number', value=10, style={'margin-bottom': '15px', 'width': '60px'}),
            html.Br(),
            html.Label('Max Steps', style={'font-weight': 'bold', 'font-size': '16px'}),
            dcc.Input(id='max-steps', type='number', value=1000, style={'margin-bottom': '15px', 'width': '80px'}),

            # Tooltip for TSP Parameters
            html.I(className="bi bi-info-circle-fill", id="tsp-info", style={'marginLeft': '5px', 'display': 'block', 'textAlign': 'center'}),
            dbc.Tooltip(
                """
                - Number of Cities: Minimum 3,these define how many nodes the TSP algorithm will optimise.
                - Episodes: Number of times the optimisation is repeated.
                - Max Steps: Maximum number of steps the optimisation will take in each episode.
                """,
                target="tsp-info",
                placement="top",
            ),
        ], style={'width': '30%', 'display': 'inline-block', 'vertical-align': 'top', 'padding': '20px',
                  'border': '1px solid #ddd', 'border-radius': '5px', 'background-color': '#f2e6ff',
                  'text-align': 'center', 'height': '250px'}),

        # Heuristics Parameters Section
        html.Div([
            html.H4('Heuristics Parameters', style={'text-align': 'center', 'font-size': '20px', 'margin-bottom': '15px'}),
            html.Label('Learning Rate', style={'font-weight': 'bold', 'font-size': '16px'}),
            dcc.Input(id='learning-rate', type='number', value=0.1, step=0.01, style={'margin-bottom': '15px', 'width': '60px'}),
            html.Br(),
            html.Label('Discount Factor', style={'font-weight': 'bold', 'font-size': '16px'}),
            dcc.Input(id='discount-factor', type='number', value=0.95, step=0.01, style={'margin-bottom': '15px', 'width': '60px'}),
            html.Br(),
            html.Label('Exploration Rate', style={'font-weight': 'bold', 'font-size': '16px'}),
            dcc.Input(id='exploration-rate', type='number', value=0.1, step=0.01, style={'margin-bottom': '15px', 'width': '60px'}),

            # Tooltip for Heuristics Parameters
            html.I(className="bi bi-info-circle-fill", id="heuristics-info", style={'marginLeft': '5px', 'display': 'block', 'textAlign': 'center'}),
            dbc.Tooltip(
                """
                - **Learning Rate**: Controls how much the agent learns from new information.
                - **Discount Factor**: Determines how much future rewards influence current decisions.
                - **Exploration Rate**: Controls the balance between exploring new strategies and exploiting known ones.
                """,
                target="heuristics-info",
                placement="top",
            ),
        ], style={'width': '30%', 'display': 'inline-block', 'vertical-align': 'top', 'padding': '20px',
                  'border': '1px solid #ddd', 'border-radius': '5px', 'background-color': '#f2e6ff',
                  'text-align': 'center', 'height': '250px'}),

        # SA Parameters Section
        html.Div([
            html.H4('SA Parameters', style={'text-align': 'center', 'font-size': '20px', 'margin-bottom': '15px'}),
            html.Label('Initial Temperature', style={'font-weight': 'bold', 'font-size': '16px'}),
            dcc.Input(id='initial-temp', type='number', value=1.0, step=0.1, style={'margin-bottom': '15px', 'width': '60px'}),
            html.Br(),
            html.Label('Temperature Decay Rate', style={'font-weight': 'bold', 'font-size': '16px'}),
            dcc.Input(id='temp-decay', type='number', value=0.99, step=0.01, style={'margin-bottom': '15px', 'width': '60px'}),

            # Tooltip for SA Parameters
            html.I(className="bi bi-info-circle-fill", id="sa-info", style={'marginLeft': '5px', 'marginTop': '40px', 'display': 'block', 'textAlign': 'center'}),
            dbc.Tooltip(
                """
                - **Initial Temperature**: Starting temperature for simulated annealing.
                - **Temperature Decay Rate**: Determines how quickly the temperature reduces over time.
                """,
                target="sa-info",
                placement="top",
            ),
        ], style={'width': '30%', 'display': 'inline-block', 'vertical-align': 'top', 'padding': '20px',
                  'border': '1px solid #ddd', 'border-radius': '5px', 'background-color': '#f2e6ff',
                  'text-align': 'center', 'height': '250px'}),
    ], style={'text-align': 'center', 'padding': '20px', 'margin-bottom': '40px'}),  # Adjust spacing between sections

    # Start Optimization Button
    html.Div([
        html.Button('Start Optimisation', id='start-button', n_clicks=0,
                    style={'background-color': '#a3d3e8', 'color': 'white', 'padding': '10px 20px',
                           'border': 'none', 'border-radius': '5px', 'font-size': '16px'}),
    ], style={'text-align': 'center', 'margin-top': '40px', 'margin-bottom': '40px'}),  # Adjust margin here

# Results Heading
    html.H3('Results', style={'textAlign': 'center', 'margin-top': '40px', 'margin-bottom': '40px'}),

        html.Div([
            html.Div([
                dcc.Graph(id='tsp-graph-geo'),
                html.I(className="bi bi-info-circle-fill", id="tsp-geo-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This graph shows the TSP route optimized using the geometric cooling schedule. The route connects all the cities in the order chosen by the optimization process. The goal is to minimize the total distance traveled.",
                    target="tsp-geo-info",
                    placement="top",
                ),
                html.P(id='geo-total-distance', style={'textAlign': 'center'}),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),

            html.Div([
                dcc.Graph(id='tsp-graph-lin'),
                html.I(className="bi bi-info-circle-fill", id="tsp-lin-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This graph shows the TSP route optimized using the linear cooling schedule. This method linearly decreases the temperature, which controls the acceptance of worse solutions over time.",
                    target="tsp-lin-info",
                    placement="top",
                ),
                html.P(id='lin-total-distance', style={'textAlign': 'center'}),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),

            html.Div([
                dcc.Graph(id='tsp-graph-lm'),
                html.I(className="bi bi-info-circle-fill", id="tsp-lm-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This graph shows the TSP route optimized using the Lundy-Mees cooling schedule. It balances exploration and exploitation by using a more complex temperature decay method.",
                    target="tsp-lm-info",
                    placement="top",
                ),
                html.P(id='lm-total-distance', style={'textAlign': 'center'}),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),
        ], style={'width': '100%', 'margin': 'auto'}),

        html.Div([
            html.Div([
                dcc.Graph(id='geo-pie'),
                html.I(className="bi bi-info-circle-fill", id="geo-pie-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This pie chart displays the utilization rate of each heuristic when the geometric cooling schedule is used. It shows how often each heuristic was selected during the optimization process.",
                    target="geo-pie-info",
                    placement="top",
                ),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),

            html.Div([
                dcc.Graph(id='lin-pie'),
                html.I(className="bi bi-info-circle-fill", id="lin-pie-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This pie chart displays the utilisation rate of each heuristic when the linear cooling schedule is used. It provides insight into the agent's decision-making process.",
                    target="lin-pie-info",
                    placement="top",
                ),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),

            html.Div([
                dcc.Graph(id='lm-pie'),
                html.I(className="bi bi-info-circle-fill", id="lm-pie-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This pie chart displays the utilisation rate of each heuristic when the Lundy-Mees cooling schedule is used. It shows the distribution of heuristic selection.",
                    target="lm-pie-info",
                    placement="top",
                ),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),
        ], style={'width': '100%', 'margin': 'auto'}),

        html.Div([
            html.Div([
                dcc.Graph(id='geo-bar'),
                html.I(className="bi bi-info-circle-fill", id="geo-bar-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This bar chart shows the effective utilisation of each heuristic under the geometric cooling schedule. It indicates how successful each heuristic was in improving the solution.",
                    target="geo-bar-info",
                    placement="top",
                ),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),

            html.Div([
                dcc.Graph(id='lin-bar'),
                html.I(className="bi bi-info-circle-fill", id="lin-bar-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This bar chart shows the effective utilisation of each heuristic under the linear cooling schedule, highlighting the impact of each heuristic on the optimization process.",
                    target="lin-bar-info",
                    placement="top",
                ),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),

            html.Div([
                dcc.Graph(id='lm-bar'),
                html.I(className="bi bi-info-circle-fill", id="lm-bar-info", style={'marginLeft': '5px'}),
                dbc.Tooltip(
                    "This bar chart shows the effective utilisation of each heuristic under the Lundy-Mees cooling schedule, reflecting how each heuristic contributed to finding the optimal solution.",
                    target="lm-bar-info",
                    placement="top",
                ),
            ], style={'display': 'inline-block', 'width': '32%', 'padding': '10px'}),
        ], style={'width': '100%', 'margin': 'auto'}),

        html.Div([
            dcc.Graph(id='reward-progress-geo'),
            html.I(className="bi bi-info-circle-fill", id="reward-progress-geo-info", style={'marginLeft': '5px'}),
            dbc.Tooltip(
                "This graph shows the reward progress for the geometric cooling schedule.",
                target="reward-progress-geo-info",
                placement="top",
            ),
            dcc.Graph(id='reward-progress-lin'),
            html.I(className="bi bi-info-circle-fill", id="reward-progress-lin-info", style={'marginLeft': '5px'}),
            dbc.Tooltip(
                "This graph shows the reward progress for the linear cooling schedule.",
                target="reward-progress-lin-info",
                placement="top",
            ),
            dcc.Graph(id='reward-progress-lm'),
            html.I(className="bi bi-info-circle-fill", id="reward-progress-lm-info", style={'marginLeft': '5px'}),
            dbc.Tooltip(
                "This graph shows the reward progress for the Lundy-Mees cooling schedule.",
                target="reward-progress-lm-info",
                placement="top",
            ),
            dcc.Graph(id='convergence-geo'),
            html.I(className="bi bi-info-circle-fill", id="convergence-geo-info", style={'marginLeft': '5px'}),
            dbc.Tooltip(
                "This graph shows the convergence progress for the geometric cooling schedule.",
                target="convergence-geo-info",
                placement="top",
            ),
            dcc.Graph(id='convergence-lin'),
            html.I(className="bi bi-info-circle-fill", id="convergence-lin-info", style={'marginLeft': '5px'}),
            dbc.Tooltip(
                "This graph shows the convergence progress for the linear cooling schedule.",
                target="convergence-lin-info",
                placement="top",
            ),
            dcc.Graph(id='convergence-lm'),
            html.I(className="bi bi-info-circle-fill", id="convergence-lm-info", style={'marginLeft': '5px'}),
            dbc.Tooltip(
                "This graph shows the convergence progress for the Lundy-Mees cooling schedule.",
                target="convergence-lm-info",
                placement="top",
            ),
        ], style={'display': 'inline-block', 'width': '100%'}),
    ])

@app.callback(
    [
        Output('tsp-graph-geo', 'figure'),
        Output('tsp-graph-lin', 'figure'),
        Output('tsp-graph-lm', 'figure'),
        Output('geo-pie', 'figure'),
        Output('lin-pie', 'figure'),
        Output('lm-pie', 'figure'),
        Output('geo-bar', 'figure'),
        Output('lin-bar', 'figure'),
        Output('lm-bar', 'figure'),
        Output('geo-total-distance', 'children'),
        Output('lin-total-distance', 'children'),
        Output('lm-total-distance', 'children'),
        Output('reward-progress-geo', 'figure'),
        Output('reward-progress-lin', 'figure'),
        Output('reward-progress-lm', 'figure'),
        Output('convergence-geo', 'figure'),
        Output('convergence-lin', 'figure'),
        Output('convergence-lm', 'figure'),
    ],
    [Input('start-button', 'n_clicks')],
    [
        State('num-cities', 'value'),
        State('episodes', 'value'),
        State('max-steps', 'value'),
        State('learning-rate', 'value'),
        State('discount-factor', 'value'),
        State('exploration-rate', 'value'),
        State('initial-temp', 'value'),
        State('temp-decay', 'value'),
    ],
)
def update_output(
    n_clicks,
    num_cities,
    episodes,
    max_steps,
    lr,
    gamma,
    epsilon,
    initial_temp,
    temp_decay,
):
    if n_clicks > 0:

        cities = np.random.rand(num_cities, 2)

        graphs = []
        pies = []
        bars = []
        distances = []
        reward_plots = []
        convergence_plots = []

        cooling_schedules = ['geometric', 'linear', 'lundy_mees']

        for schedule in cooling_schedules:
            # Passing the same set of cities to each problem domain
            problem_domain = TSPProblemDomain(cities=cities)
            agent = SimpleUtilityBasedAgent(action_size=2, problem_domain=problem_domain, adaptation_strategy='additive')

            # Train the agent and capture the metrics, with avereage distance
            convergence_data, reward_progress, avg_distance = train_utility_based_agent(agent, episodes, max_steps)

            # Create the figure for TSP route
            optimized_route = problem_domain.solutions[0]
            total_distance = calculate_total_distance(optimized_route, problem_domain.cities)


            distances.append(f"{schedule.capitalize()} - Total Distance: {total_distance:.2f} - Average Distance: {avg_distance:.2f}")

            # Create utilization figures
            utilization_rates = get_utilization_rates(agent)
            pie_chart = create_utilization_pie_chart(utilization_rates, schedule.capitalize())
            pies.append(pie_chart)

            effective_utilization_rates = get_effective_utilization_rates(agent)
            bar_chart = create_effective_utilization_bar_chart(effective_utilization_rates, schedule.capitalize())
            bars.append(bar_chart)

            # Generate reward progress plot
            reward_plot = create_reward_progress_plot(reward_progress, schedule.capitalize())
            reward_plots.append(reward_plot)

            # Generate convergence plot
            convergence_plot = create_convergence_plot(convergence_data, schedule.capitalize())
            convergence_plots.append(convergence_plot)

            graphs.append(create_tsp_figure(optimized_route, problem_domain.cities))

        return (
            graphs[0], graphs[1], graphs[2],
            pies[0], pies[1], pies[2],
            bars[0], bars[1], bars[2],
            distances[0], distances[1], distances[2],
            reward_plots[0], reward_plots[1], reward_plots[2],
            convergence_plots[0], convergence_plots[1], convergence_plots[2]
        )


    return [go.Figure()] * 9 + [""] * 3 + [go.Figure()] * 6


def get_cooling_schedule(schedule_name, initial_temp, decay_rate):
    if schedule_name == 'geometric':
        return lambda t: geometric_schedule(t, initial_temp, decay_rate)
    elif schedule_name == 'linear':
        return lambda t: linear_schedule(t, initial_temp, decay_rate)
    elif schedule_name == 'lundy_mees':
        return lambda t: lundy_mees_schedule(t, initial_temp, decay_rate)

def geometric_schedule(t, initial_temp, decay_rate):
    return initial_temp * (decay_rate ** t)

def linear_schedule(t, initial_temp, decay_rate):
    return initial_temp * (1 - t * decay_rate)

def lundy_mees_schedule(t, initial_temp, decay_rate):
    return initial_temp / (1 + t * decay_rate)

# Running the Dash server with Ngrok for external access
if __name__ == '__main__':
    ngrok.set_auth_token("2lh7SWuH9WwF1UHSUsDEtyrx7aW_4yo28yWtjCG2SNuXm7C2")
    public_url = ngrok.connect(8050)
    print("Public URL:", public_url)
    app.run_server(port=8050, debug=True)

Public URL: NgrokTunnel: "https://c818-35-231-23-143.ngrok-free.app" -> "http://localhost:8050"


<IPython.core.display.Javascript object>